In [1]:
!pip -q install -U fastapi uvicorn nest_asyncio nvidia-ml-py3 psutil pandas bitsandbytes accelerate peft transformers

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.8/111.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.3/263.3 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 19.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-adk 1.21.0 requires fastapi<0.124.0,>=0.115.0, but you have fastapi 0.126.0 which is incompatible.


In [2]:
import os, time, json, threading
import pandas as pd
import torch
import torch.nn as nn
import psutil

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
import nest_asyncio

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

from pynvml import (
    nvmlInit, nvmlShutdown, nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetPowerUsage
)

assert torch.cuda.is_available(), "GPU not detected. In Colab: Runtime > Change runtime type > GPU."
device = "cuda"
proc = psutil.Process()

print("Torch:", torch.__version__)

Torch: 2.9.0+cu126


In [3]:
from google.colab import drive
drive.mount("/content/drive")

# ---- EDIT THESE PATHS ----
PHASE2_ADAPTER_PATH = "/content/drive/My Drive/ML_Project_Phase2/phi3_squad_final"
PHASE3_HEAD_PATH    = "/content/drive/My Drive/ML_Project_Phase3/early_exit_head_block9.pt"
BASE_MODEL_ID = "microsoft/Phi-3.5-mini-instruct"

# Defaults (frontend can override per request)
DEFAULT_MAX_NEW_TOKENS = 32
DEFAULT_MAX_INPUT_TOKENS = 256

# Early-exit settings (block 9 -> 0-based index 8)
EXIT_BLOCK_INDEX = 8
DEFAULT_EE_THRESHOLD = 0.8

Mounted at /content/drive


In [4]:
# 4-bit load if bitsandbytes is available (recommended for Colab T4)
USE_4BIT = True
try:
    import bitsandbytes as bnb  # noqa
except Exception:
    USE_4BIT = False

bnb_config = None
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    print("Using 4-bit quantization (bnb).")
else:
    print("bitsandbytes not available -> using FP16 (may OOM on T4).")

# Tokenizer (use adapter path so it matches your finetuned prompting)
tokenizer = AutoTokenizer.from_pretrained(PHASE2_ADAPTER_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Base model (Phase 1)
if USE_4BIT:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        trust_remote_code=True,
        quantization_config=bnb_config,
        device_map="auto",
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        device_map="auto",
    )

# Finetuned wrapper (Phase 2 / Phase 3)
ft_model = PeftModel.from_pretrained(base_model, PHASE2_ADAPTER_PATH)

base_model.eval()
ft_model.eval()
for p in base_model.parameters():
    p.requires_grad = False
for p in ft_model.parameters():
    p.requires_grad = False

print("Loaded base + finetuned models.")

Using 4-bit quantization (bnb).


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3.5-mini-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

Loaded base + finetuned models.


In [5]:
# Early-exit head (Phase 3)
class EarlyExitHead(nn.Module):
    def __init__(self, hidden_size: int, vocab_size: int, dropout: float = 0.1):
        super().__init__()
        self.ln = nn.LayerNorm(hidden_size)
        self.ff = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, vocab_size),
        )
    def forward(self, hidden_states):
        return self.ff(self.ln(hidden_states))

def load_ee_head(path: str):
    ckpt = torch.load(path, map_location="cpu")
    head = EarlyExitHead(ckpt["hidden_size"], ckpt["vocab_size"]).to(device)
    head.load_state_dict(ckpt["head_state_dict"])
    head.eval()
    return head, ckpt

ee_head, ee_ckpt = load_ee_head(PHASE3_HEAD_PATH)
print("Loaded early-exit head. hidden_size=", ee_ckpt["hidden_size"], "vocab=", ee_ckpt["vocab_size"])

Loaded early-exit head. hidden_size= 3072 vocab= 32064


In [6]:
def unwrap_model(m):
    if hasattr(m, "base_model"):
        m = m.base_model
    if hasattr(m, "model"):
        m = m.model
    return m

def get_blocks(m):
    cands = [m, unwrap_model(m)]
    for c in cands:
        if hasattr(c, "model") and hasattr(c.model, "layers"):
            return c.model.layers
        if hasattr(c, "layers"):
            return c.layers
        if hasattr(c, "transformer") and hasattr(c.transformer, "h"):
            return c.transformer.h
        if hasattr(c, "transformer") and hasattr(c.transformer, "layers"):
            return c.transformer.layers
    raise ValueError("Could not locate transformer blocks.")

def forward_until_block_hidden(model, input_ids, attention_mask, block_index: int):
    blocks = get_blocks(model)
    assert 0 <= block_index < len(blocks)

    class _Grab(Exception):
        def __init__(self, hs): self.hs = hs

    def hook_fn(mod, inputs, output):
        hs = output[0] if isinstance(output, (tuple, list)) else output
        raise _Grab(hs)

    h = blocks[block_index].register_forward_hook(hook_fn)
    try:
        with torch.inference_mode():
            _ = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False, output_hidden_states=False)
    except _Grab as e:
        return e.hs
    finally:
        h.remove()
    raise RuntimeError("Hidden state not captured.")

def greedy_next_token_from_logits(logits):
    probs = torch.softmax(logits, dim=-1)
    conf, pred = probs.max(dim=-1)
    return pred, float(conf.item())

# End token ids (stop generation when produced)
END_IDS = set()
if tokenizer.eos_token_id is not None:
    END_IDS.add(int(tokenizer.eos_token_id))
for t in ["<|end|>", "<|endoftext|>"]:
    tid = tokenizer.convert_tokens_to_ids(t)
    if tid is not None and tid != tokenizer.unk_token_id:
        END_IDS.add(int(tid))

print("END_IDS:", END_IDS)

END_IDS: {32000, 32007}


In [7]:
def _nvml_energy_mj(handle):
    from pynvml import nvmlDeviceGetTotalEnergyConsumption
    return nvmlDeviceGetTotalEnergyConsumption(handle)

def _nvml_power_w(handle):
    return nvmlDeviceGetPowerUsage(handle) / 1000.0

class StopAfterBlock(Exception):
    pass

def profile_one_forward_blocks(model, input_ids, attention_mask, stop_after_block_index=None):
    blocks = get_blocks(model)
    stats = {}

    nvmlInit()
    handle = nvmlDeviceGetHandleByIndex(0)

    energy_counter_ok = True
    try:
        _ = _nvml_energy_mj(handle)
    except Exception:
        energy_counter_ok = False

    handles = []

    def make_hooks(layer_idx):
        state = {}
        def pre_hook(mod, inputs):
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
            state["t0"] = time.perf_counter()
            state["rss0_mb"] = proc.memory_info().rss / (1024**2)
            if energy_counter_ok:
                state["e0_mj"] = _nvml_energy_mj(handle)
            else:
                state["p0_w"] = _nvml_power_w(handle)

        def post_hook(mod, inputs, output):
            torch.cuda.synchronize()
            t1 = time.perf_counter()
            dt = t1 - state["t0"]

            rss1_mb = proc.memory_info().rss / (1024**2)
            cpu_rss_mb = max(state["rss0_mb"], rss1_mb)

            gpu_peak_alloc_mb = torch.cuda.max_memory_allocated() / (1024**2)

            if energy_counter_ok:
                e1_mj = _nvml_energy_mj(handle)
                energy_j = max(0.0, (e1_mj - state["e0_mj"]) / 1000.0)
            else:
                p1_w = _nvml_power_w(handle)
                energy_j = max(0.0, 0.5 * (state.get("p0_w", p1_w) + p1_w) * dt)

            stats[layer_idx] = {
                "time_ms": dt * 1000.0,
                "energy_j": float(energy_j),
                "gpu_peak_alloc_MB": float(gpu_peak_alloc_mb),
                "cpu_rss_MB": float(cpu_rss_mb),
            }

        return pre_hook, post_hook

    for i, blk in enumerate(blocks):
        pre, post = make_hooks(i)
        handles.append(blk.register_forward_pre_hook(pre))
        handles.append(blk.register_forward_hook(post))

    stop_handle = None
    if stop_after_block_index is not None:
        def stop_hook(mod, inputs, output):
            raise StopAfterBlock()
        stop_handle = blocks[stop_after_block_index].register_forward_hook(stop_hook)

    try:
        with torch.inference_mode():
            _ = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False, output_hidden_states=False)
    except StopAfterBlock:
        pass
    finally:
        for h in handles:
            h.remove()
        if stop_handle is not None:
            stop_handle.remove()
        nvmlShutdown()

    return stats, energy_counter_ok

In [8]:
def run_inference(mode: str, prompt: str, max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
                  max_input_tokens: int = DEFAULT_MAX_INPUT_TOKENS, ee_threshold: float = DEFAULT_EE_THRESHOLD,
                  include_token_log: bool = True):
    assert mode in {"pretrained", "finetuned", "early_exit"}

    model = base_model if mode == "pretrained" else ft_model

    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_input_tokens)
    input_ids = enc["input_ids"].to(device)
    attention_mask = enc.get("attention_mask", None)
    if attention_mask is not None:
        attention_mask = attention_mask.to(device)

    # Warmup
    with torch.inference_mode():
        _ = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)

    n_layers = len(get_blocks(model))
    agg = [{
        "layer": i,
        "executed_steps": 0,
        "total_time_ms": 0.0,
        "total_energy_j": 0.0,
        "gpu_peak_alloc_MB": 0.0,
        "cpu_rss_MB": 0.0,
    } for i in range(n_layers)]

    steps_exited = 0
    steps_full = 0
    energy_counter_ok_final = None

    step_log = []
    generated_token_ids = []

    for step in range(int(max_new_tokens)):
        stop_after = None
        exited = False
        conf = None
        next_tok = None

        if mode == "early_exit":
            hs = forward_until_block_hidden(ft_model, input_ids, attention_mask, block_index=EXIT_BLOCK_INDEX)
            hs_dtype = hs.dtype
            head = ee_head.to(device=device, dtype=hs_dtype)

            with torch.no_grad():
                last_h = hs[:, -1, :].clone().to(dtype=hs_dtype)
                ee_logits = head(last_h)
                next_tok_ee, conf = greedy_next_token_from_logits(ee_logits)

            if conf >= float(ee_threshold):
                stop_after = EXIT_BLOCK_INDEX
                exited = True
                steps_exited += 1
                next_tok = next_tok_ee
            else:
                steps_full += 1

        layer_stats, energy_counter_ok = profile_one_forward_blocks(
            model, input_ids, attention_mask, stop_after_block_index=stop_after
        )
        energy_counter_ok_final = energy_counter_ok_final if energy_counter_ok_final is not None else energy_counter_ok

        if not exited:
            with torch.no_grad():
                out = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
                last_logits = out.logits[:, -1, :]
                next_tok, conf2 = greedy_next_token_from_logits(last_logits)
                if mode != "early_exit":
                    conf = conf2

        # Aggregate stats
        for i in range(n_layers):
            s = layer_stats.get(i)
            if s is None:
                continue
            agg[i]["executed_steps"] += 1
            agg[i]["total_time_ms"] += float(s["time_ms"])
            agg[i]["total_energy_j"] += float(s["energy_j"])
            agg[i]["gpu_peak_alloc_MB"] = max(agg[i]["gpu_peak_alloc_MB"], float(s["gpu_peak_alloc_MB"]))
            agg[i]["cpu_rss_MB"] = max(agg[i]["cpu_rss_MB"], float(s["cpu_rss_MB"]))

        if include_token_log:
            step_log.append({
                "step": int(step),
                "exited_early": bool(exited),
                "confidence": float(conf) if conf is not None else None,
                "input_len": int(input_ids.shape[1]),
                "next_token_id": int(next_tok.item()),
                "next_token_text": tokenizer.decode(next_tok),
            })

        generated_token_ids.append(int(next_tok.item()))

        # Append token
        input_ids = torch.cat([input_ids, next_tok.view(1, 1)], dim=1)
        if attention_mask is not None:
            attention_mask = torch.cat([attention_mask, torch.ones((1,1), device=device, dtype=attention_mask.dtype)], dim=1)

        # Stop on end token
        if int(next_tok.item()) in END_IDS:
            break

    if mode != "early_exit":
        steps_exited = 0
        steps_full = len(generated_token_ids)

    layer_df = pd.DataFrame(agg)
    token_df = pd.DataFrame(step_log)

    answer_text = tokenizer.decode(generated_token_ids, skip_special_tokens=True).strip()

    result = {
        "mode": mode,
        "prompt": prompt,
        "answer_text": answer_text,
        "token_count_total": int(len(generated_token_ids)),
        "early_exit": {
            "enabled": (mode == "early_exit"),
            "steps_exited": int(steps_exited),
            "steps_full": int(steps_full),
            "exit_block_index": int(EXIT_BLOCK_INDEX),
            "threshold": float(ee_threshold),
        },
        "profiling": {
            "energy_counter_ok": bool(energy_counter_ok_final),
            "units": {
                "total_time_ms": "ms",
                "total_energy_j": "J",
                "gpu_peak_alloc_MB": "MB",
                "cpu_rss_MB": "MB",
            },
            "layers": layer_df.to_dict(orient="records"),
        },
        "token_log": token_df.to_dict(orient="records") if include_token_log else [],
    }
    return result

In [9]:
app = FastAPI(title="Phi-3 Phase 5 Backend", version="1.0")

app.add_middleware(
    CORSMiddleware,
    allow_origin_regex=r"^https://.*-colab\.googleusercontent\.com$",
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class InferRequest(BaseModel):
    mode: str                 # "pretrained" | "finetuned" | "early_exit"
    prompt: str
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS
    max_input_tokens: int = DEFAULT_MAX_INPUT_TOKENS
    ee_threshold: float = DEFAULT_EE_THRESHOLD
    include_token_log: bool = True

@app.get("/health")
def health():
    return {"ok": True}

@app.post("/infer")
def infer(req: InferRequest):
    mode = req.mode.strip().lower()
    if mode not in {"pretrained", "finetuned", "early_exit"}:
        return {"error": "Invalid mode", "allowed": ["pretrained", "finetuned", "early_exit"]}
    return run_inference(
        mode=mode,
        prompt=req.prompt,
        max_new_tokens=req.max_new_tokens,
        max_input_tokens=req.max_input_tokens,
        ee_threshold=req.ee_threshold,
        include_token_log=req.include_token_log,
    )

In [10]:
from fastapi.responses import HTMLResponse

@app.get("/", response_class=HTMLResponse)
def home():
    return """
<!doctype html>
<html>
<head>
  <meta charset="utf-8" />
  <title>Phi-3 Phase 5 Demo</title>
  <style>
    body { font-family: system-ui; max-width: 980px; margin: 20px auto; }
    textarea, input, select, button { font-family: inherit; }
    textarea { width: 100%; padding: 8px; }
    input, select { width: 100%; padding: 8px; }
    .row { display:flex; gap:12px; }
    .col { flex:1; }
    pre { background:#0b1220; color:#e5e7eb; padding:10px; border-radius:8px; white-space:pre-wrap; }
    #status { background:#111827; }
    .btn { padding:10px 14px; }
    .btn2 { padding:8px 12px; }

    /* Modal */
    .modal-backdrop{
      position:fixed; inset:0;
      background:rgba(0,0,0,0.45);
      display:none; align-items:center; justify-content:center;
      padding:16px;
    }
    .modal{
      width:min(980px,98vw);
      max-height:88vh;
      background:#fff;
      border-radius:12px;
      overflow:hidden;
      box-shadow:0 10px 40px rgba(0,0,0,0.25);
      display:flex; flex-direction:column;
    }
    .modal-header{
      padding:12px 14px;
      display:flex; justify-content:space-between; align-items:center;
      border-bottom:1px solid #e5e7eb;
    }
    .modal-body{ padding:12px 14px; overflow:auto; }
    table{ width:100%; border-collapse:collapse; font-size:13px; }
    th, td{ border-bottom:1px solid #e5e7eb; padding:6px 8px; }
    th{ background:#f3f4f6; text-align:right; position:sticky; top:0; }
    td.num{ text-align:right; }
    td.txt{ text-align:left; }
  </style>
</head>
<body>
  <h2>Phi-3 Phase 5 Demo</h2>

  <label><b>Mode</b></label>
  <select id="mode">
    <option value="pretrained">pretrained</option>
    <option value="finetuned">finetuned</option>
    <option value="early_exit">early_exit</option>
  </select>

  <div class="row" style="margin-top:10px;">
    <div class="col">
      <label><b>max_new_tokens</b></label>
      <input id="max_new_tokens" type="number" value="32"/>
    </div>
    <div class="col">
      <label><b>EE threshold</b></label>
      <input id="ee_threshold" type="number" step="0.01" value="0.8"/>
    </div>
  </div>

  <label style="margin-top:10px; display:block;"><b>Prompt</b></label>
  <textarea id="prompt" placeholder="Enter your prompt here" rows="6"></textarea>

  <div style="margin-top:10px; display:flex; gap:10px; align-items:center;">
    <button id="run" class="btn">Run inference</button>
    <label style="display:flex; gap:6px; align-items:center;">
      <input id="include_token_log" style="width:fit-content" type="checkbox" checked />
      Include token log
    </label>
  </div>

  <pre id="status" style="margin-top:10px;">Ready.</pre>

  <h3>Answer</h3>
  <pre id="answer"></pre>

  <div id="stats" style="margin-top:8px;"></div>

  <!-- THESE are the triggers for the modal -->
  <div style="margin-top:10px; display:flex; gap:10px; flex-wrap:wrap;">
    <button id="btnLayers" class="btn2" disabled>View Layer Table</button>
    <button id="btnTokens" class="btn2" disabled>View Token Log</button>
  </div>

  <!-- Modal -->
  <div id="backdrop" class="modal-backdrop" onclick="hideModal()">
    <div class="modal" onclick="event.stopPropagation()">
      <div class="modal-header">
        <div id="modalTitle" style="font-weight:600;">Table</div>
        <button class="btn2" onclick="hideModal()">Close</button>
      </div>
      <div class="modal-body">
        <div id="modalContent"></div>
      </div>
    </div>
  </div>

<script>
let lastLayers = [];
let lastTokens = [];

function setStatus(msg){ document.getElementById("status").textContent = msg; }

function fmt3(x){ return (typeof x === "number") ? (Math.round(x*1000)/1000) : x; }

function buildTable(rows, cols, leftCols=new Set()){
  if(!rows || rows.length===0) return "<div>No data.</div>";
  const thead = "<tr>" + cols.map(c=>`<th>${c}</th>`).join("") + "</tr>";
  const tbody = rows.map(r=>{
    const tds = cols.map(c=>{
      const v = (r[c]===null || r[c]===undefined) ? "" : r[c];
      const cls = leftCols.has(c) ? "txt" : "num";
      return `<td class="${cls}">${v}</td>`;
    }).join("");
    return `<tr>${tds}</tr>`;
  }).join("");
  return `<table><thead>${thead}</thead><tbody>${tbody}</tbody></table>`;
}

function showModal(title, html){
  document.getElementById("modalTitle").textContent = title;
  document.getElementById("modalContent").innerHTML = html;
  document.getElementById("backdrop").style.display = "flex";
}
function hideModal(){ document.getElementById("backdrop").style.display = "none"; }

document.getElementById("btnLayers").onclick = () => {
  const cols = ["layer","executed_steps","total_time_ms","total_energy_j","gpu_peak_alloc_MB","cpu_rss_MB"];
  const rows = (lastLayers||[]).map(r=>{
    const o={}; for(const c of cols) o[c]=fmt3(r[c]); return o;
  });
  showModal("Layer-wise Profiling Table", buildTable(rows, cols));
};

document.getElementById("btnTokens").onclick = () => {
  const cols = ["step","exited_early","confidence","input_len","next_token_id","next_token_text"];
  const left = new Set(["exited_early","next_token_text"]);
  const rows = (lastTokens||[]).map(r=>{
    const o={}; for(const c of cols) o[c]=fmt3(r[c]); return o;
  });
  showModal("Token Log (early_exit)", buildTable(rows, cols, left));
};

document.getElementById("run").onclick = async () => {
  setStatus("Calling /infer ...");
  document.getElementById("answer").textContent = "";
  document.getElementById("stats").innerHTML = "";
  document.getElementById("btnLayers").disabled = true;
  document.getElementById("btnTokens").disabled = true;

  const payload = {
    mode: document.getElementById("mode").value,
    prompt: document.getElementById("prompt").value,
    max_new_tokens: parseInt(document.getElementById("max_new_tokens").value || "32", 10),
    ee_threshold: parseFloat(document.getElementById("ee_threshold").value || "0.8"),
    include_token_log: document.getElementById("include_token_log").checked
  };

  const res = await fetch("/infer", {
    method:"POST",
    headers:{ "Content-Type":"application/json" },
    body: JSON.stringify(payload)
  });

  const data = await res.json();
  if(data.error){
    setStatus("Error: " + JSON.stringify(data));
    return;
  }

  document.getElementById("answer").textContent = data.answer_text || "";
  const ee = data.early_exit || {};
  const prof = data.profiling || {};

  lastLayers = (prof.layers || []);
  lastTokens = (data.token_log || []);

  document.getElementById("btnLayers").disabled = !(lastLayers.length);
  document.getElementById("btnTokens").disabled = !(lastTokens.length);

  document.getElementById("stats").innerHTML =
    `<div><b>Mode:</b> ${data.mode}</div>` +
    `<div><b>Total tokens:</b> ${data.token_count_total}</div>` +
    `<div><b>Early-exit:</b> ${ee.enabled ? "ON" : "OFF"} | exited steps: ${ee.steps_exited} | full steps: ${ee.steps_full}</div>` +
    `<div><b>Energy counter ok:</b> ${prof.energy_counter_ok}</div>`;

  setStatus("Done.");
};
</script>
</body>
</html>
"""


In [11]:
nest_asyncio.apply()

PORT = 8085

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(1.5)
print("Server started on port", PORT)

from google.colab import output
public_url = output.eval_js(f"google.colab.kernel.proxyPort({PORT})")
print("Backend URL:", public_url)

Server started on port 8085
Backend URL: https://8085-gpu-t4-s-pcjeatzinlit-b.us-west1-0.prod.colab.dev
